In [66]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [67]:
df = pd.read_csv('C:/Users/daryn/Downloads/garments_worker_productivity.csv')

In [68]:
df.head()

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [69]:
df.shape

(1197, 15)

In [70]:
df.dtypes

date                         str
quarter                      str
department                   str
day                          str
team                       int64
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                  int64
incentive                  int64
idle_time                float64
idle_men                   int64
no_of_style_change         int64
no_of_workers            float64
actual_productivity      float64
dtype: object

In [71]:
df['date'] = pd.to_datetime(df['date']).dt.year.astype(float)

In [72]:
features = [
    'date',
    'targeted_productivity', 
    'incentive', 
    'smv', 
    'no_of_style_change'
]

X = df[features]
y = df['actual_productivity']

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [74]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[ 0. , 0.71, 0. ,-0. ,-0.04]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['date','targeted_productivity','incentive','smv','no_of_style_change']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.2345
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(4)


In [75]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [76]:
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-8))) * 100

In [77]:
def calculate_adjusted_r2(r2, n, p):
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))

In [78]:
def get_metrics_dict(y_true, y_pred, n, p):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = calculate_adjusted_r2(r2, n, p)
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape,
        'R²': r2,
        'Adjusted R²': adj_r2
    }

In [80]:
n_train = len(y_train)
n_test = len(y_test)
p = X.shape[1]

train_metrics = get_metrics_dict(y_train, y_train_pred, n_train, p)
test_metrics = get_metrics_dict(y_test, y_test_pred, n_test, p)

metrics_table = pd.DataFrame({'Train': train_metrics, 'Test': test_metrics})

print("=== Metrics ===")
display(metrics_table)

test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_test_pred})
test_results['Absolute Error'] = np.abs(test_results['Actual'] - test_results['Predicted'])
top_5_errors = test_results.sort_values(by='Absolute Error', ascending=False).head(5)

print("\n=== Top 5 errors ===")
display(top_5_errors)

=== Metrics ===


,Train,Test
MSE,0.024883,0.022214
RMSE,0.157745,0.149045
MAE,0.111149,0.106337
MAPE (%),19.981339,17.841525
R²,0.206432,0.163376
Adjusted R²,0.202260,0.145499



=== Top 5 errors ===


,Actual,Predicted,Absolute Error
715,0.994375,0.478840,0.515535
984,0.271875,0.763361,0.491486
298,0.313853,0.797683,0.483830
783,0.287042,0.762256,0.475214
920,0.325000,0.727050,0.402050


In [84]:
close_to_zero = y_test[y_test < 0.1]

print("Number of target values close to 0:", len(close_to_zero))
print(close_to_zero)

Number of target values close to 0: 0
Series([], Name: actual_productivity, dtype: float64)


In [81]:
results = []

for i in range(1, len(features) + 1):

    current_features = features[:i]

    X = df[current_features]
    y = df['actual_productivity']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    n = len(y_test)
    p = len(current_features)

    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    results.append({
        'Features': ', '.join(current_features),
        'Number of features': p,
        'R2': r2,
        'Adjusted R2': adjusted_r2
    })

results_df = pd.DataFrame(results)

display(results_df)

,Features,Number of features,R2,Adjusted R2
0,date,1,-0.006425,-0.010653
1,"date, targeted_productivity",2,0.135216,0.127919
2,"date, targeted_productivity, incentive",3,0.142684,0.131786
3,"date, targeted_productivity, incentive, smv",4,0.155731,0.141361
4,"date, targeted_productivity, incentive, smv, n...",5,0.163376,0.145499


In [83]:
comparison_table = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_test_pred
})

print(comparison_table)

       Actual  Predicted
0    0.268214   0.595818
1    0.800359   0.793966
2    0.681061   0.798788
3    0.325000   0.727050
4    0.667604   0.798788
..        ...        ...
235  0.700170   0.720034
236  0.971867   0.691588
237  0.950625   0.656196
238  0.715767   0.727015
239  0.800003   0.734238

[240 rows x 2 columns]


### Analysis Answers

1. **What do RMSE and MAE mean in the units of your target?**
   - MAE means that, on average, our productivity predictions are off by about 0.106. RMSE is 0.149, which gives slightly more weight to larger errors.

2. **Is the model overfitting? Compare train and test.**
   - No, there is no overfitting. The test performance ($R^2 \approx 0.163$, $\text{MAE} \approx 0.106$) is very close to and slightly better than the train performance.

3. **Is there a big gap between MAE and RMSE? Show the 5 largest errors. What do these observations have in common?**
   - The gap is moderate. The largest errors typically occur at extreme actual productivity values (very low or very high values) where the linear model struggles to fit.

4. **Is MAPE reliable for your data? Check for target values close to 0.**
   - MAPE is relatively stable because the minimum target value in this dataset is around 0.23 (no true zeros), avoiding division-by-zero issues.

5. **Add features one by one and track Adjusted R². Which features are worth keeping?**
   - Core production features like `targeted_productivity`, `incentive`, `smv`, and `no_of_style_change` improve the adjusted R² and are worth keeping.

---

### Conclusion
- **Which one metric would you report for your dataset, and why?**
  - We would report **MAE** because it directly represents the average error in clear, interpretable units of worker productivity.
- **Was your guess from the pitch correct?**
  - Yes, our initial guess that MAE would be the most intuitive and practical metric for production evaluation was fully correct.